# DTEN AI & Machine Learning Internship — Task 1

## Adaptive Prototype–Residual Voting for Dry Bean Classification

This project uses the Kaggle Dry Bean Dataset to classify seven varieties of dry beans.

The proposed model is called Adaptive Prototype–Residual Voting, or APRV.

APRV is a project-specific, interpretable classification algorithm that combines:

1. Robust feature scaling.
2. Class-based prototypes.
3. Feature reliability weights.
4. Prototype residual distances.
5. Adaptive nearest-neighbor voting.
6. Class-prior correction.

Dataset link:

https://www.kaggle.com/datasets/muratkokludataset/dry-bean-dataset

Importing Libraries

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## Dataset

The Dry Bean Dataset contains numerical measurements of dry beans.

The target variable is the bean variety. The dataset contains seven classes:

- Seker
- Barbunya
- Bombay
- Cali
- Dermosan
- Horoz
- Sira

The classification problem is multiclass because the model must choose one class from seven possible classes.

Downloading the Kaggle Dataset

In [ ]:
import kagglehub

DATASET_HANDLE = "muratkokludataset/dry-bean-dataset"

dataset_dir = kagglehub.dataset_download(DATASET_HANDLE)

print("Dataset downloaded successfully.")
print("Dataset directory:")
print(dataset_dir)

all_files = glob.glob(
    os.path.join(dataset_dir, "**", "*"),
    recursive=True
)

print("\nFiles found:")
for file in all_files[:20]:
    print(file)

Loading the Dataset

In [ ]:
# Searching for Excel and CSV files
excel_files = glob.glob(
    os.path.join(dataset_dir, "**", "*.xlsx"),
    recursive=True
)

csv_files = glob.glob(
    os.path.join(dataset_dir, "**", "*.csv"),
    recursive=True
)

data_files = excel_files + csv_files

if len(data_files) == 0:
    raise FileNotFoundError(
        "No CSV or XLSX file was found in the Kaggle dataset folder."
    )

file_path = data_files[0]

print("Using file:")
print(file_path)

if file_path.lower().endswith(".xlsx"):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path)

# Cleaning column names
df.columns = [str(column).strip() for column in df.columns]

print("\nDataset shape:")
print(df.shape)

print("\nFirst five rows:")
display(df.head())

print("\nColumn names:")
print(list(df.columns))

Inspecting and Cleaning the Data

In [ ]:
print("Dataset information:")
display(df.info())

print("\nMissing values per column:")
display(df.isnull().sum())

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())

print("\nDataset summary:")
display(df.describe(include="all"))

In [ ]:
# Identifying the target column
TARGET = "Class"

if TARGET not in df.columns:
    possible_targets = [
        column for column in df.columns
        if column.lower() == "class"
    ]

    if len(possible_targets) == 0:
        raise KeyError(
            f"Target column 'Class' was not found. "
            f"Available columns are: {list(df.columns)}"
        )

    TARGET = possible_targets[0]

print("Target column:", TARGET)

# Display class distribution
print("\nClass distribution:")
display(df[TARGET].value_counts())

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("\nShape after removing duplicate rows:")
print(df.shape)

***Preparing Features and labels***

In [ ]:
# Separating input features and target
X_df = df.drop(columns=[TARGET])
y_text = df[TARGET].astype(str)

# Converting all feature columns to numerical values
X_df = X_df.apply(pd.to_numeric, errors="raise")

feature_names = X_df.columns.tolist()

# Encoding class labels as integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

# Converting features to NumPy format
X = X_df.to_numpy(dtype=float)

classes = label_encoder.classes_

print("Number of observations:", X.shape[0])
print("Number of features:", X.shape[1])
print("Feature names:")
print(feature_names)

print("\nEncoded classes:")
for number, class_name in enumerate(classes):
    print(number, "=", class_name)

## Preprocessing strategy

The data is divided into training and testing sets using an 80:20 stratified split.

Stratification ensures that the training and testing sets preserve approximately the same proportion of each bean class.

The model learns the median and interquartile range only from the training data. This prevents test-set information from leaking into the training process.

Robust scaling is used because it is less affected by unusually large or small measurements than ordinary standardization.

***Splitting and robustly scaling the data***

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

# Calculate robust scaling values using training data only
train_median = np.median(X_train, axis=0)

q1 = np.percentile(X_train, 25, axis=0)
q3 = np.percentile(X_train, 75, axis=0)

train_iqr = q3 - q1

# Prevent division by zero
train_iqr[train_iqr < 1e-9] = 1.0

# Apply robust scaling
X_train_scaled = (X_train - train_median) / train_iqr
X_test_scaled = (X_test - train_median) / train_iqr

print("Robust scaling completed.")

## APRV algorithm

For each bean class, APRV calculates a prototype using the median value of every feature.

The algorithm then calculates feature reliability. A feature receives a larger weight when:

- The class prototypes are far apart on that feature.
- The classes have relatively small internal variation on that feature.

For a new observation, APRV calculates its residual distance from every class prototype.

The model also calculates a local vote from nearby training observations. The local vote becomes weaker when the new observation is far away from known training examples.

The final prediction combines:

1. Prototype-based evidence.
2. Adaptive local voting.
3. A small class-prior correction.

***Defining the APRV Algorithm***

In [ ]:
class AdaptivePrototypeResidualVoting:
    """
    Adaptive Prototype–Residual Voting classifier.

    This is a project-specific interpretable classifier for
    numerical multiclass tabular data.
    """

    def __init__(
        self,
        n_neighbors=9,
        local_strength=0.25,
        prior_strength=0.02
    ):
        self.n_neighbors = n_neighbors
        self.local_strength = local_strength
        self.prior_strength = prior_strength

    def fit(self, X, y):
        """
        Train the APRV classifier.
        """

        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)

        self.X_train_ = X
        self.y_train_ = y

        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]


        self.prototypes_ = np.vstack([
            np.median(X[y == class_id], axis=0)
            for class_id in self.classes_
        ])


        spreads = []

        for class_id in self.classes_:
            class_data = X[y == class_id]
            class_prototype = self.prototypes_[class_id]

            absolute_residuals = np.abs(
                class_data - class_prototype
            )

            median_absolute_deviation = np.median(
                absolute_residuals,
                axis=0
            )

            robust_spread = (
                1.4826 * median_absolute_deviation
            ) + 0.05

            spreads.append(robust_spread)

        self.spreads_ = np.vstack(spreads)
        global_center = np.median(X, axis=0)

        between_class_separation = np.mean(
            (self.prototypes_ - global_center) ** 2,
            axis=0
        )

        within_class_variation = np.mean(
            self.spreads_,
            axis=0
        )

        raw_feature_reliability = (
            between_class_separation /
            (within_class_variation ** 2 + 1e-9)
        )

        # Normalize feature weights so that they sum to one
        self.feature_weights_ = (
            raw_feature_reliability + 1e-6
        ) / (
            raw_feature_reliability.sum()
            + 1e-6 * len(raw_feature_reliability)
        )


        self.priors_ = np.array([
            np.mean(y == class_id)
            for class_id in self.classes_
        ])

        return self

    def _calculate_prototype_distances(self, X):


        X = np.asarray(X, dtype=float)

        residuals = np.abs(
            X[:, None, :]
            - self.prototypes_[None, :, :]
        )

        scaled_residuals = (
            residuals /
            self.spreads_[None, :, :]
        )

        weighted_distances = np.sum(
            scaled_residuals
            * self.feature_weights_[None, None, :],
            axis=2
        )

        return weighted_distances

    def predict_proba(self, X):


        X = np.asarray(X, dtype=float)


        prototype_distances = (
            self._calculate_prototype_distances(X)
        )

        prototype_evidence = np.exp(
            -prototype_distances
        )

        prototype_evidence = (
            prototype_evidence /
            prototype_evidence.sum(axis=1, keepdims=True)
        )


        pairwise_distances = np.sqrt(
            (
                (
                    X[:, None, :]
                    - self.X_train_[None, :, :]
                ) ** 2
            ).sum(axis=2)
        )

        number_of_neighbors = min(
            self.n_neighbors,
            len(self.X_train_)
        )

        nearest_indices = np.argpartition(
            pairwise_distances,
            kth=number_of_neighbors - 1,
            axis=1
        )[:, :number_of_neighbors]

        nearest_distances = np.take_along_axis(
            pairwise_distances,
            nearest_indices,
            axis=1
        )

        # Inverse-distance voting
        neighbor_weights = 1.0 / (
            nearest_distances + 1e-6
        )

        local_votes = np.zeros(
            (len(X), len(self.classes_))
        )

        for row_index in range(len(X)):
            neighbor_labels = self.y_train_[
                nearest_indices[row_index]
            ]

            for class_position, class_id in enumerate(
                self.classes_
            ):
                class_mask = (
                    neighbor_labels == class_id
                )

                local_votes[
                    row_index,
                    class_position
                ] = neighbor_weights[
                    row_index
                ][class_mask].sum()

        local_votes = (
            local_votes /
            local_votes.sum(axis=1, keepdims=True)
        )


        # Adaptive local-vote strength

        closest_distance = np.min(
            nearest_distances,
            axis=1
        )

        isolation_factor = np.exp(
            -closest_distance
        )

        adaptive_strength = (
            self.local_strength
            * isolation_factor
        )


        # Combining global and local evidence

        combined_scores = (
            (1 - adaptive_strength[:, None])
            * prototype_evidence
            + adaptive_strength[:, None]
            * local_votes
        )

        # Small class-prior adjustment
        combined_scores = (
            combined_scores
            + self.prior_strength
            * np.log(self.priors_ + 1e-12)[None, :]
        )

        # Preventing zero values
        combined_scores = np.clip(
            combined_scores,
            1e-12,
            None
        )

        # Normalizing into probabilities
        probabilities = (
            combined_scores /
            combined_scores.sum(
                axis=1,
                keepdims=True
            )
        )

        return probabilities

    def predict(self, X):



        probabilities = self.predict_proba(X)

        prediction_positions = np.argmax(
            probabilities,
            axis=1
        )

        return self.classes_[
            prediction_positions
        ]

    def explain_one(self, x):


        x = np.asarray(x, dtype=float)
        x = x.reshape(1, -1)

        distances = (
            self._calculate_prototype_distances(x)[0]
        )

        explanation = pd.DataFrame({
            "class": label_encoder.inverse_transform(
                self.classes_
            ),
            "prototype_distance": distances,
            "prototype_evidence": np.exp(-distances)
        })

        return explanation.sort_values(
            by="prototype_distance"
        )

***Training APRV***

In [ ]:
aprv_model = AdaptivePrototypeResidualVoting(
    n_neighbors=9,
    local_strength=0.25,
    prior_strength=0.02
)

aprv_model.fit(
    X_train_scaled,
    y_train
)

print("APRV model trained successfully.")

***Inspecting Feature Reliability***

In [ ]:
feature_weight_table = pd.DataFrame({
    "feature": feature_names,
    "reliability_weight": aprv_model.feature_weights_
})

feature_weight_table = feature_weight_table.sort_values(
    by="reliability_weight",
    ascending=False
)

display(feature_weight_table)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_weight_table,
    x="reliability_weight",
    y="feature",
    palette="viridis"
)

plt.title("APRV Feature Reliability Weights")
plt.xlabel("Reliability weight")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

***Generating Predictions***

In [ ]:
y_pred = aprv_model.predict(X_test_scaled)

print("Predictions generated.")
print("Number of predictions:", len(y_pred))

***Calculating required evaluation metrics***

In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("APRV Evaluation Results")
print("-----------------------")
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro-Precision: {macro_precision:.4f}")
print(f"Macro-Recall:    {macro_recall:.4f}")
print(f"Macro-F1:        {macro_f1:.4f}")

In [ ]:
print("Detailed Classification Report")
print("-------------------------------")

classification_results = classification_report(
    y_test,
    y_pred,
    target_names=classes,
    zero_division=0
)

print(classification_results)

***Confusion Matrix***

In [ ]:
confusion = confusion_matrix(
    y_test,
    y_pred
)

confusion_table = pd.DataFrame(
    confusion,
    index=classes,
    columns=classes
)

print("Confusion Matrix Values:")
display(confusion_table)

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    confusion_table,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.title("APRV Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

***Optional Comparison with Random Forest***

In [ ]:


random_forest = RandomForestClassifier(
    n_estimators=250,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest.fit(
    X_train_scaled,
    y_train
)

random_forest_pred = random_forest.predict(
    X_test_scaled
)

random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_pred
)

random_forest_precision = precision_score(
    y_test,
    random_forest_pred,
    average="macro",
    zero_division=0
)

random_forest_recall = recall_score(
    y_test,
    random_forest_pred,
    average="macro",
    zero_division=0
)

random_forest_f1 = f1_score(
    y_test,
    random_forest_pred,
    average="macro",
    zero_division=0
)

comparison_table = pd.DataFrame({
    "Model": [
        "APRV Proposed Model",
        "Random Forest Baseline"
    ],
    "Accuracy": [
        accuracy,
        random_forest_accuracy
    ],
    "Macro Precision": [
        macro_precision,
        random_forest_precision
    ],
    "Macro Recall": [
        macro_recall,
        random_forest_recall
    ],
    "Macro F1": [
        macro_f1,
        random_forest_f1
    ]
})

display(comparison_table.round(4))

***Explaining one prediction***

In [ ]:
sample_position = 0

actual_class = classes[y_test[sample_position]]
predicted_class = classes[y_pred[sample_position]]

print("Actual class:", actual_class)
print("Predicted class:", predicted_class)

print("\nPrototype explanation:")
display(
    aprv_model.explain_one(
        X_test_scaled[sample_position]
    )
)

***Saving the evaluation results***

In [ ]:
metrics_table = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Value": [
        accuracy,
        macro_precision,
        macro_recall,
        macro_f1
    ]
})

display(metrics_table.round(4))

metrics_table.to_csv(
    "aprv_metrics.csv",
    index=False
)

confusion_table.to_csv(
    "aprv_confusion_matrix.csv"
)

feature_weight_table.to_csv(
    "aprv_feature_weights.csv",
    index=False
)

print("The following files were saved:")
print("- aprv_metrics.csv")
print("- aprv_confusion_matrix.csv")
print("- aprv_feature_weights.csv")

## Results Interpretation

After running the notebook, replace the bracketed values below with the actual values printed by the model.

The proposed Adaptive Prototype–Residual Voting model achieved an accuracy of **[insert accuracy]**, macro-precision of **[insert macro-precision]**, macro-recall of **[insert macro-recall]**, and macro-F1 of **[insert macro-F1]** on the held-out test set.

The accuracy indicates the overall percentage of correctly classified beans. Macro-precision gives equal importance to every bean variety and measures how reliable the predictions are for each class. Macro-recall measures how many examples from each class were successfully identified. Macro-F1 provides a balanced summary of precision and recall.

The confusion matrix should be examined to identify the classes that are most frequently confused. These errors may occur because some bean varieties have similar geometric and morphological measurements.

The main advantage of APRV is interpretability. The prediction is based on class prototypes, feature reliability, and local residual voting. A limitation is that APRV is designed for numerical tabular data and can be affected by the scaling method, neighbor count, and train/test split.

A stronger future experiment could use repeated stratified cross-validation, parameter tuning, probability calibration, and an independent external dataset.

***The project developed an original project-specific classifier called Adaptive Prototype–Residual Voting, or APRV, for classifying dry bean varieties. The algorithm combines global class prototypes with adaptive local voting. Unlike a simple nearest-neighbor method, APRV assigns reliability weights to features according to their class separation and internal stability. It also reduces the influence of local voting when a test observation is far from the training data. The model was evaluated using accuracy, macro-precision, macro-recall, macro-F1, and a confusion matrix. This approach is transparent because each prediction can be explained using prototype distances, feature weights, and local class evidence.***